# 04. Early Test-Set Check — LightGBM and Logistic Regression baselines

> ⚠️ **This is an early check, NOT the final evaluation.**
>
> The project rules say the test set should be used once, at the end, for the final chosen model.
> This notebook looks at the test set **before** tuning and model selection, only to check the baselines.
> - Record this in the change log (date + reason).
> - Do **not** use these test scores to choose models, features, hyperparameters or thresholds.
> - Mention in the report that the test set was viewed early.

What this notebook does:
1. Trains the same two baseline models as `04d_lightgbm_baseline.ipynb` and `04a_logistic_regression_baseline.ipynb` (default settings, no preprocessing, **train.csv only**).
2. Scores each model on the **validation** set and the **test** set.
3. Shows both scores side by side, to check whether validation was a good estimate of unseen data.

Positive class = abnormal (`figo_label = 1`). Threshold = 0.5 (default).

## Step 1: Setup and imports

In [1]:
import sys
sys.path.append('../..')

import pandas as pd
from lightgbm import LGBMClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score
from sklearn.metrics import f1_score, roc_auc_score, balanced_accuracy_score, confusion_matrix

from config import config
from src.data_preprocessing import get_feature_columns

## Step 2: Load the train, validation and test sets

In [2]:
train_df = pd.read_csv(config.PROCESSED_TRAIN_FILE)
val_df = pd.read_csv(config.PROCESSED_VAL_FILE)
test_df = pd.read_csv(config.PROCESSED_TEST_FILE)

print("Train shape:     ", train_df.shape)
print("Validation shape:", val_df.shape)
print("Test shape:      ", test_df.shape)

Train shape:      (14540, 106)
Validation shape: (3115, 106)
Test shape:       (3114, 106)


## Step 3: Separate features (X) and target (y)

In [3]:
# Get the 103 signal features (IDs and target are excluded)
feature_cols = get_feature_columns(train_df)

X_train = train_df[feature_cols]
y_train = train_df[config.TARGET_COLUMN]

X_val = val_df[feature_cols]
y_val = val_df[config.TARGET_COLUMN]

X_test = test_df[feature_cols]
y_test = test_df[config.TARGET_COLUMN]

print("Train:", X_train.shape, " Validation:", X_val.shape, " Test:", X_test.shape)

Number of feature columns found: 103
Train: (14540, 103)  Validation: (3115, 103)  Test: (3114, 103)


## Step 4: Safety check — no patient appears in more than one set

In [4]:
train_patients = set(train_df[config.PATIENT_ID_COLUMN])
val_patients = set(val_df[config.PATIENT_ID_COLUMN])
test_patients = set(test_df[config.PATIENT_ID_COLUMN])

train_and_test = train_patients.intersection(test_patients)
val_and_test = val_patients.intersection(test_patients)

print("Patients in both train and test:     ", len(train_and_test))
print("Patients in both validation and test:", len(val_and_test))

if len(train_and_test) > 0 or len(val_and_test) > 0:
    raise ValueError("Patient overlap found. Stop and investigate the split.")

print("Check passed: test patients are not in train or validation.")

Patients in both train and test:      0
Patients in both validation and test: 0
Check passed: test patients are not in train or validation.


## Step 5: Helper function to calculate the metrics

In [5]:
def get_metrics(model, X, y):
    """Predict with the model and return a dictionary of metrics (abnormal = 1 is positive)."""
    # Class predictions (threshold 0.5) and probability of abnormal (1)
    y_pred = model.predict(X)
    y_prob = model.predict_proba(X)[:, 1]

    metrics = {}
    metrics["Accuracy"] = round(accuracy_score(y, y_pred), 4)
    metrics["Precision"] = round(precision_score(y, y_pred), 4)
    metrics["Recall"] = round(recall_score(y, y_pred), 4)
    metrics["F1-score"] = round(f1_score(y, y_pred), 4)
    metrics["ROC-AUC"] = round(roc_auc_score(y, y_prob), 4)
    metrics["Balanced accuracy"] = round(balanced_accuracy_score(y, y_pred), 4)

    print("Confusion matrix [[TN, FP], [FN, TP]]:")
    print(confusion_matrix(y, y_pred))
    return metrics

## Step 6: LightGBM — train on train.csv, score on validation and test

In [6]:
# Same settings as 04d_lightgbm_baseline.ipynb (defaults, seed 42)
lgbm_model = LGBMClassifier(random_state=config.RANDOM_STATE)
lgbm_model.fit(X_train, y_train)

print("LightGBM - Validation")
lgbm_val = get_metrics(lgbm_model, X_val, y_val)

print("\nLightGBM - Test")
lgbm_test = get_metrics(lgbm_model, X_test, y_test)

[LightGBM] [Info] Number of positive: 7821, number of negative: 6719
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.008456 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 22881
[LightGBM] [Info] Number of data points in the train set: 14540, number of used features: 103
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.537895 -> initscore=0.151873
[LightGBM] [Info] Start training from score 0.151873
LightGBM - Validation
Confusion matrix [[TN, FP], [FN, TP]]:
[[1329  110]
 [ 109 1567]]

LightGBM - Test
Confusion matrix [[TN, FP], [FN, TP]]:
[[1344   96]
 [ 101 1573]]


## Step 7: Logistic Regression — train on train.csv, score on validation and test

Same as `04a_logistic_regression_baseline.ipynb`: no scaling, so a `ConvergenceWarning` is expected.

In [7]:
# Same settings as 04a_logistic_regression_baseline.ipynb (defaults, seed 42)
lr_model = LogisticRegression(random_state=config.RANDOM_STATE)
lr_model.fit(X_train, y_train)

print("Logistic Regression - Validation")
lr_val = get_metrics(lr_model, X_val, y_val)

print("\nLogistic Regression - Test")
lr_test = get_metrics(lr_model, X_test, y_test)

Logistic Regression - Validation
Confusion matrix [[TN, FP], [FN, TP]]:
[[1022  417]
 [ 477 1199]]

Logistic Regression - Test
Confusion matrix [[TN, FP], [FN, TP]]:
[[1025  415]
 [ 454 1220]]


c:\Users\Rathishan\Desktop\Desktop\Git\ML_Module\Machine-Learning-Based-Fetal-Well-being-Classification-Using-Cardiotocography-Features\venv\Lib\site-packages\sklearn\linear_model\_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 100 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=100).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


## Step 8: Side-by-side comparison

If validation and test scores are close, the validation set was a good estimate of performance on unseen patients.

In [8]:
comparison = pd.DataFrame({
    "LightGBM (val)": lgbm_val,
    "LightGBM (test)": lgbm_test,
    "LogReg (val)": lr_val,
    "LogReg (test)": lr_test,
})

print(comparison)

                   LightGBM (val)  LightGBM (test)  LogReg (val)  \
Accuracy                   0.9297           0.9367        0.7130   
Precision                  0.9344           0.9425        0.7420   
Recall                     0.9350           0.9397        0.7154   
F1-score                   0.9347           0.9411        0.7284   
ROC-AUC                    0.9844           0.9865        0.7863   
Balanced accuracy          0.9293           0.9365        0.7128   

                   LogReg (test)  
Accuracy                  0.7209  
Precision                 0.7462  
Recall                    0.7288  
F1-score                  0.7374  
ROC-AUC                   0.7885  
Balanced accuracy         0.7203  
